# OULAD — Step 1: Data Ingestion and Cleaning

## 1. Load all CSVs
- reads all CSV files from oulad/ folder
- applies basic_clean() to every table
- basic_clean: strips column names, removes duplicates, strips string whitespace

In [2]:
import pandas as pd
import os

path = "oulad/"

def basic_clean(df):
    df.columns = df.columns.str.strip().str.lower()
    df = df.drop_duplicates()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    return df

assessments         = basic_clean(pd.read_csv(path + "assessments.csv"))
courses             = basic_clean(pd.read_csv(path + "courses.csv"))
studentAssessment   = basic_clean(pd.read_csv(path + "studentAssessment.csv"))
studentInfo         = basic_clean(pd.read_csv(path + "studentInfo.csv"))
studentRegistration = basic_clean(pd.read_csv(path + "studentRegistration.csv"))
studentVle          = basic_clean(pd.read_csv(path + "studentVle.csv"))
vle                 = basic_clean(pd.read_csv(path + "vle.csv"))

print("All tables loaded and cleaned")

All tables loaded and cleaned


## 2. Reusable inspection function
- check() shows shape, dtypes, describe, missing values
- run this on any table before cleaning

In [3]:
def check(df):
    print(f"Shape:\n{df.shape}\n")
    print(f"Dtypes:\n{df.dtypes}\n")
    print(f"Describe:\n{df.describe()}\n")
    print(f"Missing Values:\n{df.isnull().sum()}\n")

## 3. courses — cleaning
- no issues found

In [4]:
# check(courses)

## 4. assessments — cleaning
- date column had "?" values causing object dtype
- fix: pd.to_numeric(errors="coerce") converts ? to NaN

In [5]:
#explore
# check(assessments)
assessments.head(10)
assessments['date'].unique()
assessments["date"] = pd.to_numeric(assessments["date"],errors="coerce").astype('Int64')

## 5. vle — cleaning
- week_from and week_to had high null % and not needed for KPIs
- action: both columns dropped

In [6]:
#explore
# check(vle)
vle[vle["week_from"]=="?"].shape
vle[vle["week_to"]=="?"].shape
vle =  vle.drop(columns=["week_from","week_to"])

## 6. studentRegistration — cleaning
- date_registration and date_unregistration had "?" causing object dtype
- fix: pd.to_numeric(errors="coerce") on both columns
- NaN in date_unregistration = student never unregistered (meaningful, left as-is)

In [7]:
# check(studentRegistration)
studentRegistration["date_registration"] = pd.to_numeric(studentRegistration["date_registration"],errors="coerce").astype('Int64')
studentRegistration["date_unregistration"] = pd.to_numeric(studentRegistration["date_unregistration"],errors="coerce").astype('Int64')

## 7. studentInfo — cleaning
- imd_band had "?" values
- fix: replace("?", None)

In [8]:
# check(studentInfo)
studentInfo["imd_band"] = studentInfo["imd_band"].replace("?",None)

## 8. studentAssessment — cleaning
- score column had "?" causing object dtype
- fix: pd.to_numeric(errors="coerce")
- NaN score = student did not submit (meaningful)

In [9]:
# check(studentAssessment)
studentAssessment["score"].unique()
studentAssessment["score"] = pd.to_numeric(studentAssessment["score"],errors="coerce")

## 9. studentVle — cleaning
- no issues found

In [10]:
# check(studentVle)

In [11]:
dim_enrollment = studentInfo  # all columns, all rows, no dedup

In [16]:
studentInfo = studentInfo[[
    "id_student",
    "gender",
    "region",
    "imd_band",
    "age_band",
    "disability",
    "highest_education"
]].drop_duplicates(subset="id_student")

## 10. Save cleaned tables to disk
- cleaned DataFrames saved to cleaned/ folder for use in next notebook
- cleaning changes exist only in memory — saving ensures they are not lost
- raw oulad/ folder is never modified

In [17]:
tables = {
    "assessments": assessments,
    "courses": courses,
    "studentAssessment": studentAssessment,
    "studentRegistration": studentRegistration,
    "studentInfo": studentInfo,
    "dim_enrollment": dim_enrollment,
    "studentVle": studentVle,
    "vle": vle
}

os.makedirs("cleaned", exist_ok=True)

for name, df in tables.items():
    df.to_csv(f"cleaned/{name}.csv", index=False)
    print(f"{name} saved")

assessments saved
courses saved
studentAssessment saved
studentRegistration saved
studentInfo saved
dim_enrollment saved
studentVle saved
vle saved
